In [13]:
!uv pip install -q sentence-transformers scikit-learn

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


class retriver:
    """
    Simple Retrieval System

    Required methods:
        1. chunker()
        2. embedder()
        3. search()
    """

    def __init__(
        self,
        document,
        chunk_size=500,
        overlap=50,
        model_name="all-MiniLM-L6-v2"
    ):
        """
        Initialize the retriever.

        Parameters
        ----------
        document : str
            Original document.
        chunk_size : int
            Maximum size of each chunk.
        overlap : int
            Number of characters shared between neighboring chunks.
        model_name : str
            Sentence Transformer model used for embeddings.
        """

        if not isinstance(document, str):
            raise TypeError("document must be a string.")

        if not document.strip():
            raise ValueError("document cannot be empty.")

        if chunk_size <= 0:
            raise ValueError("chunk_size must be greater than 0.")

        if overlap < 0:
            raise ValueError("overlap cannot be negative.")

        if overlap >= chunk_size:
            raise ValueError(
                "overlap must be smaller than chunk_size."
            )

        self.document = document.strip()

        self.chunk_size = chunk_size
        self.overlap = overlap

        self.chunks = []
        self.embeddings = None

        # Embedding model
        self.model = SentenceTransformer(model_name)

    # ---------------------------------------------------------
    # 1. CHUNKER
    # ---------------------------------------------------------

    def chunker(self):
        """
        Split the document into overlapping chunks.

        Returns
        -------
        list
            List of text chunks.
        """

        self.chunks = []

        start = 0
        document_length = len(self.document)

        while start < document_length:

            end = start + self.chunk_size

            chunk = self.document[start:end].strip()

            if chunk:
                self.chunks.append(chunk)

            # Move forward while keeping overlap
            start = end - self.overlap

        return self.chunks

    # ---------------------------------------------------------
    # 2. EMBEDDER
    # ---------------------------------------------------------

    def embedder(self):
        """
        Convert document chunks into numerical vectors.

        Returns
        -------
        numpy.ndarray
            Embedding matrix.
        """

        # If chunking has not been performed yet,
        # automatically create chunks.
        if not self.chunks:
            self.chunker()

        self.embeddings = self.model.encode(
            self.chunks,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        return self.embeddings

    # ---------------------------------------------------------
    # 3. SEARCH
    # ---------------------------------------------------------

    def search(self, query, top_k=3):
        """
        Search for chunks most similar to the query.

        Parameters
        ----------
        query : str
            User's search query.

        top_k : int
            Number of results to return.

        Returns
        -------
        list of dictionaries
            Relevant chunks and similarity scores.
        """

        if not isinstance(query, str):
            raise TypeError("query must be a string.")

        if not query.strip():
            raise ValueError("query cannot be empty.")

        if top_k <= 0:
            raise ValueError("top_k must be greater than 0.")

        # Create embeddings if they don't already exist.
        if self.embeddings is None:
            self.embedder()

        # Convert query into an embedding.
        query_embedding = self.model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        # Calculate similarity between query and every chunk.
        scores = cosine_similarity(
            query_embedding,
            self.embeddings
        )[0]

        # Don't ask for more results than we have chunks.
        top_k = min(top_k, len(self.chunks))

        # Get indexes of highest scores.
        top_indices = np.argsort(scores)[-top_k:][::-1]

        results = []

        for rank, index in enumerate(top_indices, start=1):

            results.append({
                "rank": rank,
                "chunk_index": int(index),
                "score": float(scores[index]),
                "text": self.chunks[index]
            })

        return results

    # ---------------------------------------------------------
    # OPTIONAL DISPLAY METHOD
    # ---------------------------------------------------------

    def show_results(self, query, top_k=3):
        """
        Print search results in a readable format.
        """

        results = self.search(query, top_k)

        print("=" * 70)
        print(f"QUERY: {query}")
        print("=" * 70)

        for result in results:

            print(f"\nRank       : {result['rank']}")
            print(f"Chunk      : {result['chunk_index']}")
            print(f"Similarity : {result['score']:.4f}")
            print(f"Text       : {result['text']}")

            print("-" * 70)

In [15]:
document = """
Machine learning is a branch of artificial intelligence that allows
computers to learn patterns from data without being explicitly programmed
for every individual task.

Supervised learning is a type of machine learning where the model learns
from labeled training data. Classification and regression are common
examples of supervised learning.

Unsupervised learning works with unlabeled data. It attempts to discover
hidden structures, patterns, or relationships in the data. Clustering is
a common example of unsupervised learning.

Deep learning is a subfield of machine learning that uses artificial
neural networks containing multiple layers. Deep learning has achieved
excellent results in image recognition, speech recognition, and natural
language processing.

Natural language processing, commonly called NLP, focuses on enabling
computers to understand, process, and generate human language.

Computer vision is an area of artificial intelligence that enables
computers to interpret and understand information from images and videos.

Reinforcement learning is a machine learning approach in which an agent
learns by interacting with an environment and receiving rewards or
penalties for its actions.
"""

In [16]:
r = retriver(
    document=document,
    chunk_size=300,
    overlap=50
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow


In [18]:

import numpy as np
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
embeddings = r.embedder()

print("Embedding shape:", embeddings.shape)

In [ ]:
r.show_results(
    "What is deep learning?",
    top_k=3
)

In [ ]:
class network:
    """
    Neural Network for MNIST handwritten digit classification.

    The model contains exactly 4 layers inside Sequential.

    Custom methods:
        - train()
        - predict()
    """

    def __init__(self):

        # -----------------------------------------------------
        # 4 LAYERS
        # -----------------------------------------------------

        self.model = Sequential([

            # Layer 1
            layers.Flatten(
                input_shape=(28, 28)
            ),

            # Layer 2
            layers.Dense(
                128,
                activation="relu"
            ),

            # Layer 3
            layers.Dense(
                64,
                activation="relu"
            ),

            # Layer 4
            layers.Dense(
                10,
                activation="softmax"
            )
        ])

        # -----------------------------------------------------
        # COMPILE MODEL
        # -----------------------------------------------------

        self.model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

    # ---------------------------------------------------------
    # CUSTOM TRAINING METHOD
    # ---------------------------------------------------------

    def train(
        self,
        x_train,
        y_train,
        epochs=5,
        batch_size=32,
        validation_split=0.1
    ):
        """
        Train the neural network.
        """

        history = self.model.fit(
            x_train,
            y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            verbose=1
        )

        return history

    # ---------------------------------------------------------
    # CUSTOM PREDICTION METHOD
    # ---------------------------------------------------------

    def predict(self, x):
        """
        Predict digit classes.

        Returns
        -------
        numpy.ndarray
            Predicted digit labels.
        """

        probabilities = self.model.predict(
            x,
            verbose=0
        )

        predictions = np.argmax(
            probabilities,
            axis=1
        )

        return predictions

    # ---------------------------------------------------------
    # OPTIONAL EVALUATION METHOD
    # ---------------------------------------------------------

    def evaluate(self, x_test, y_test):
        """
        Evaluate model performance.
        """

        loss, accuracy = self.model.evaluate(
            x_test,
            y_test,
            verbose=0
        )

        print(f"Test Loss     : {loss:.4f}")
        print(f"Test Accuracy : {accuracy:.4f}")

        return loss, accuracy

    # ---------------------------------------------------------
    # OPTIONAL SUMMARY
    # ---------------------------------------------------------

    def summary(self):
        """
        Display model architecture.
        """

        return self.model.summary()

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

In [ ]:
net = network()
net.summary()

In [ ]:
history = net.train(
    x_train,
    y_train,
    epochs=5,
    batch_size=32
)

In [ ]:
net.evaluate(
    x_test,
    y_test
)

In [ ]:
predictions = net.predict(
    x_test[:10]
)

print("Predicted:", predictions)
print("Actual   :", y_test[:10])